In [1]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

# Connect to database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='',
    database='healthcare_db'
)

# Helper function — runs SQL and returns a dataframe
def run_query(sql):
    return pd.read_sql(sql, conn)

print("Connected and ready!")

Connected and ready!


In [3]:
q1 = """
SELECT 
    medical_condition,
    COUNT(*) as patient_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 2) as percentage
FROM patients
GROUP BY medical_condition
ORDER BY patient_count DESC
"""

df_q1 = run_query(q1)
print(df_q1)

  medical_condition  patient_count  percentage
0         Arthritis           9308       16.77
1          Diabetes           9304       16.76
2      Hypertension           9245       16.66
3           Obesity           9231       16.63
4            Cancer           9227       16.63
5            Asthma           9185       16.55


In [2]:
from sqlalchemy import create_engine

# Create SQLAlchemy engine (cleaner connection for pandas)
engine = create_engine('mysql+mysqlconnector://root:@localhost/healthcare_db')

# Updated helper function
def run_query(sql):
    return pd.read_sql(sql, engine)

print("Connected with SQLAlchemy!")

Connected with SQLAlchemy!


In [ ]:
import sys
!{sys.executable} -m pip install sqlalchemy

In [4]:
q2 = """
SELECT 
    medical_condition,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(MIN(billing_amount), 2) as min_billing,
    ROUND(MAX(billing_amount), 2) as max_billing
FROM patients
GROUP BY medical_condition
ORDER BY avg_billing DESC
"""

df_q2 = run_query(q2)
print(df_q2)

  medical_condition  avg_billing  min_billing  max_billing
0           Obesity     25805.97     -1310.27     52024.73
1          Diabetes     25638.41     -1316.62     52211.85
2            Asthma     25635.25     -1520.42     52181.84
3         Arthritis     25497.33     -1130.00     52170.04
4      Hypertension     25497.10     -1660.01     52764.28
5            Cancer     25161.79     -2008.49     52373.03


In [5]:
q3 = """
SELECT 
    admission_type,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 2) as percentage
FROM patients
GROUP BY admission_type
ORDER BY patient_count DESC
"""

df_q3 = run_query(q3)
print(df_q3)

  admission_type  patient_count  avg_billing  percentage
0       Elective          18655     25602.23       33.61
1         Urgent          18576     25517.36       33.47
2      Emergency          18269     25497.40       32.92


In [6]:
q4 = """
SELECT 
    insurance_provider,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(SUM(billing_amount), 2) as total_billing
FROM patients
GROUP BY insurance_provider
ORDER BY total_billing DESC
"""

df_q4 = run_query(q4)
print(df_q4)

  insurance_provider  patient_count  avg_billing  total_billing
0              Cigna          11249     25525.77   2.871393e+08
1           Medicare          11154     25615.99   2.857208e+08
2         Blue Cross          11059     25613.01   2.832543e+08
3   UnitedHealthcare          11125     25389.17   2.824545e+08
4              Aetna          10913     25553.29   2.788631e+08


In [7]:
q5 = """
SELECT 
    test_results,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 2) as percentage
FROM patients
GROUP BY test_results
ORDER BY count DESC
"""

df_q5 = run_query(q5)
print(df_q5)

   test_results  count  percentage
0      Abnormal  18627       33.56
1        Normal  18517       33.36
2  Inconclusive  18356       33.07


In [8]:
q6 = """
SELECT 
    hospital,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(AVG(age), 1) as avg_patient_age
FROM patients
GROUP BY hospital
HAVING patient_count > 100
ORDER BY avg_billing DESC
LIMIT 10
"""

df_q6 = run_query(q6)
print(df_q6)

Empty DataFrame
Columns: [hospital, patient_count, avg_billing, avg_patient_age]
Index: []


In [9]:
q7 = """
SELECT 
    CASE 
        WHEN age < 18 THEN 'Under 18'
        WHEN age BETWEEN 18 AND 35 THEN '18-35'
        WHEN age BETWEEN 36 AND 50 THEN '36-50'
        WHEN age BETWEEN 51 AND 65 THEN '51-65'
        ELSE 'Over 65'
    END as age_group,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    medical_condition
FROM patients
GROUP BY age_group, medical_condition
ORDER BY age_group, patient_count DESC
"""

df_q7 = run_query(q7)
print(df_q7.head(20))

   age_group  patient_count  avg_billing medical_condition
0      18-35           2422     25963.11           Obesity
1      18-35           2419     25986.09            Asthma
2      18-35           2413     25346.60            Cancer
3      18-35           2408     25258.55         Arthritis
4      18-35           2384     25401.85      Hypertension
5      18-35           2370     25482.82          Diabetes
6      36-50           2115     25440.44          Diabetes
7      36-50           2084     25344.22         Arthritis
8      36-50           2047     26245.85           Obesity
9      36-50           2035     25686.35      Hypertension
10     36-50           2011     24855.74            Cancer
11     36-50           2009     25556.09            Asthma
12     51-65           2122     25664.51           Obesity
13     51-65           2107     25914.47          Diabetes
14     51-65           2072     25702.37            Cancer
15     51-65           2071     25621.12      Hypertensi

In [11]:
q6 = """
SELECT 
    hospital,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(AVG(age), 1) as avg_patient_age
FROM patients
GROUP BY hospital
ORDER BY patient_count DESC
LIMIT 10
"""

df_q6 = run_query(q6)
print(df_q6)

      hospital  patient_count  avg_billing  avg_patient_age
0    LLC Smith             44     23413.41             54.3
1    Ltd Smith             39     25727.32             56.3
2  Johnson PLC             38     28531.65             46.0
3    Smith Ltd             37     26217.19             53.3
4    Smith PLC             36     28595.12             42.3
5  Smith Group             36     22406.42             47.8
6  Johnson Inc             35     26889.07             50.1
7    Smith Inc             34     22895.07             53.6
8    Smith LLC             32     22865.11             51.9
9  Group Smith             32     28217.99             47.6


In [12]:
q8 = """
SELECT 
    medical_condition,
    ROUND(AVG(DATEDIFF(discharge_date, date_of_admission)), 1) as avg_stay_days,
    MIN(DATEDIFF(discharge_date, date_of_admission)) as min_stay,
    MAX(DATEDIFF(discharge_date, date_of_admission)) as max_stay
FROM patients
GROUP BY medical_condition
ORDER BY avg_stay_days DESC
"""

df_q8 = run_query(q8)
print(df_q8)

  medical_condition  avg_stay_days  min_stay  max_stay
0            Asthma           15.7         1        30
1            Cancer           15.5         1        30
2      Hypertension           15.5         1        30
3           Obesity           15.5         1        30
4         Arthritis           15.5         1        30
5          Diabetes           15.4         1        30


In [13]:
q9 = """
SELECT 
    medical_condition,
    medication,
    COUNT(*) as prescription_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY medical_condition), 2) as pct_of_condition
FROM patients
GROUP BY medical_condition, medication
ORDER BY medical_condition, prescription_count DESC
"""

df_q9 = run_query(q9)
print(df_q9)

   medical_condition   medication  prescription_count  pct_of_condition
0          Arthritis      Aspirin                1918             20.61
1          Arthritis  Paracetamol                1877             20.17
2          Arthritis   Penicillin                1866             20.05
3          Arthritis      Lipitor                1825             19.61
4          Arthritis    Ibuprofen                1822             19.57
5             Asthma  Paracetamol                1888             20.56
6             Asthma   Penicillin                1845             20.09
7             Asthma    Ibuprofen                1827             19.89
8             Asthma      Lipitor                1823             19.85
9             Asthma      Aspirin                1802             19.62
10            Cancer      Lipitor                1922             20.83
11            Cancer    Ibuprofen                1873             20.30
12            Cancer  Paracetamol                1853           

In [14]:
q10 = """
SELECT 
    blood_type,
    COUNT(*) as patient_count,
    ROUND(AVG(billing_amount), 2) as avg_billing,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM patients), 2) as percentage
FROM patients
GROUP BY blood_type
ORDER BY patient_count DESC
"""

df_q10 = run_query(q10)
print(df_q10)

  blood_type  patient_count  avg_billing  percentage
0         A-           6969     25595.02       12.56
1         A+           6956     25664.57       12.53
2        AB+           6947     25361.46       12.52
3        AB-           6945     25694.93       12.51
4         B+           6945     25429.72       12.51
5         B-           6944     25524.42       12.51
6         O+           6917     25249.74       12.46
7         O-           6877     25795.66       12.39
